# Fetch Affected CVES from National Vulnerbaility Database On the basis of keyword. Only Run to Inspect CVE's Related to a Specific Keyword, not to fetch actual device CVE. 

+ Use identified CPE name the file fetchcves-using-cpe.ipynb to fetch device specific CVEs
+ The CPE list fetched by keyword is visible at the end of this notebook



In [ ]:
import requests
import json
import os
import time
from datetime import datetime

In [ ]:
keyword = "infusion pump"
exact_match = False          # True: adds bare keywordExactMatch flag
results_per_page = 2000      # max value
output_dir = "KeywordCVEsMatch"
progress_file = "progress.json"
log_file = os.path.join(output_dir, "fetch_log.txt")

os.makedirs(output_dir, exist_ok=True)

print("Keyword:", keyword)
print("Exact match:", exact_match)
print("Results per page:", results_per_page)
print("Output dir:", os.path.abspath(output_dir))

In [ ]:
def fetch_cves_by_keyword(keyword="infusion pump", exact_match=False, start_index=0, results_per_page=2000):
    """
    Fetch CVEs from the NVD API filtered by keyword search.
    keywordExactMatch is a flag parameter with no value: include it when exact_match is True.
    """
    base_url = "https://services.nvd.nist.gov/rest/json/cves/2.0"

    params = {
        "keywordSearch": keyword,
        "startIndex": start_index,
        "resultsPerPage": results_per_page,
    }
    if exact_match:
        params["keywordExactMatch"] = ""

    time.sleep(6)  # respect NVD rate limiting

    headers = {
        "User-Agent": "CVE-Fetcher/1.0 (Research Purpose)"
    }

    try:
        response = requests.get(base_url, params=params, headers=headers, timeout=60)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 429:
            print("Rate limit exceeded. Waiting 30 seconds before retrying...")
            time.sleep(30)
            return fetch_cves_by_keyword(keyword, exact_match, start_index, results_per_page)
        else:
            print(f"Error fetching data: {response.status_code}")
            print(f"Response: {response.text}")
            return None
    except Exception as e:
        print(f"Exception occurred: {e}")
        time.sleep(30)
        return fetch_cves_by_keyword(keyword, exact_match, start_index, results_per_page)

def extract_cpes(cve):
    """
    Walk configurations -> nodes (possibly nested via 'children') -> cpeMatch
    and return a list of dicts with CPE details for this CVE.
    """
    cpe_entries = []

    def walk_nodes(nodes):
        for node in nodes:
            for match in node.get("cpeMatch", []):
                cpe_entries.append({
                    "criteria": match.get("criteria"),
                    "vulnerable": match.get("vulnerable"),
                    "versionStartIncluding": match.get("versionStartIncluding"),
                    "versionEndIncluding": match.get("versionEndIncluding"),
                    "versionEndExcluding": match.get("versionEndExcluding"),
                })
            # some records nest further via 'children'
            if "children" in node:
                walk_nodes(node["children"])

    for config in cve.get("configurations", []):
        walk_nodes(config.get("nodes", []))

    return cpe_entries


def parse_cpe_string(criteria):
    """
    Break a cpe:2.3:... string into vendor/product for readability.
    cpe:2.3:a:hospira:mednet:*:*:*:*:*:*:*:*
              ^part ^vendor ^product
    """
    if not criteria:
        return None, None
    parts = criteria.split(":")
    if len(parts) >= 5:
        return parts[3], parts[4]  # vendor, product
    return None, None

In [ ]:
def save_cve_to_file(KeywordCVEsMatch):
    cve_id = KeywordCVEsMatch.get("id", "unknown")
    filename = f"{cve_id}.json".replace('/', '_')
    file_path = os.path.join(output_dir, filename)
    with open(file_path, 'w') as f:
        json.dump(KeywordCVEsMatch, f, indent=2)
    return file_path

In [ ]:
start_index = 0
total_saved = 0

if os.path.exists(progress_file):
    with open(progress_file, 'r') as f:
        progress_data = json.load(f)
        start_index = progress_data.get("next_start_index", 0)
        total_saved = progress_data.get("total_saved", 0)
        print(f"Resuming from index {start_index}, already saved {total_saved} CVEs")
else:
    print("No progress file found: starting fresh from index 0")

In [ ]:
start_time = datetime.now()

while True:
    print(f"Fetching CVEs (keyword='{keyword}', exact={exact_match}) from index {start_index}...")
    response = fetch_cves_by_keyword(
        keyword=keyword,
        exact_match=exact_match,
        start_index=start_index,
        results_per_page=results_per_page
    )

    if not response:
        print("Failed to fetch CVEs, retrying in 60 seconds...")
        time.sleep(60)
        continue

    total_results = response.get("totalResults", 0)
    results_per_page = response.get("resultsPerPage", results_per_page)
    vulnerabilities = response.get("vulnerabilities", [])
    batch_count = len(vulnerabilities)

    if batch_count == 0:
        print("No more CVEs to fetch.")
        break

    print(f"Total CVEs available matching keyword: {total_results}")
    print(f"Processing {batch_count} vulnerabilities...")

    with open(log_file, 'a') as log:
        log.write(f"[{datetime.now()}] Fetching batch starting at index {start_index}, containing {batch_count} CVEs\n")

    current_batch_saved = 0
    for vuln in vulnerabilities:
        cve_item = vuln.get("cve", {})
        save_cve_to_file(cve_item)
        current_batch_saved += 1
        if current_batch_saved % 100 == 0:
            print(f"Saved {current_batch_saved}/{batch_count} CVEs in current batch")

    total_saved += current_batch_saved
    next_start_index = start_index + batch_count

    with open(progress_file, 'w') as f:
        json.dump({
            "next_start_index": next_start_index,
            "total_saved": total_saved,
            "total_results": total_results,
            "keyword": keyword,
            "exact_match": exact_match,
            "last_update": datetime.now().isoformat()
        }, f, indent=2)

    if next_start_index >= total_results:
        print(f"Completed fetching all {total_results} CVEs for keyword '{keyword}'.")
        break

    start_index = next_start_index

    progress_percentage = (total_saved / total_results) * 100 if total_results else 0
    elapsed_time = (datetime.now() - start_time).total_seconds() / 60
    estimated_total_time = (elapsed_time / progress_percentage) * 100 if progress_percentage > 0 else 0
    estimated_remaining_time = estimated_total_time - elapsed_time

    print(f"Overall progress: {total_saved}/{total_results} CVEs ({progress_percentage:.2f}%)")
    print(f"Elapsed time: {elapsed_time:.2f} minutes")
    print(f"Estimated remaining time: {estimated_remaining_time:.2f} minutes")
    print(" ")

end_time = datetime.now()
total_time = (end_time - start_time).total_seconds() / 60
print(f"Completed saving {total_saved} CVEs in {total_time:.2f} minutes.")
with open(log_file, 'a') as log:
    log.write(f"[{datetime.now()}] Completed fetching all {total_saved} CVEs in {total_time:.2f} minutes.\n")

In [ ]:
saved_files = sorted(f for f in os.listdir(output_dir) if f.endswith(".json"))

print("Total CVE files saved:", len(saved_files))
print("\nFirst 10:")
for f in saved_files[:10]:
    print(" ", f)

total_size_mb = sum(os.path.getsize(os.path.join(output_dir, f)) for f in saved_files) / (1024 * 1024)
print(f"\nTotal size on disk: {total_size_mb:.2f} MB")

In [ ]:
# Progress.json progress record must be deleted for new start
example_file = saved_files[0]
with open(os.path.join(output_dir, example_file), "r") as f:
    example_cve = json.load(f)

print("File:", example_file)
print(json.dumps(example_cve, indent=2))

In [ ]:
import pandas as pd

summary_rows = []

for fname in saved_files:
    with open(os.path.join(output_dir, fname), "r") as f:
        cve = json.load(f)

    cve_id = cve.get("id", "unknown")

    published = cve.get("published", "")
    year = published[:4] if published else "unknown"

    metrics = cve.get("metrics", {})
    base_score = None
    for metric_key in ("cvssMetricV31", "cvssMetricV30", "cvssMetricV2"):
        if metric_key in metrics and metrics[metric_key]:
            base_score = metrics[metric_key][0]["cvssData"].get("baseScore")
            break

    descriptions = cve.get("descriptions", [])
    en_desc = next((d["value"] for d in descriptions if d.get("lang") == "en"), "")

    cpes = extract_cpes(cve)
    vulnerable_cpes = [c["criteria"] for c in cpes if c.get("vulnerable")]

    vendors, products = set(), set()
    for c in cpes:
        vendor, product = parse_cpe_string(c["criteria"])
        if vendor:
            vendors.add(vendor)
        if product:
            products.add(product)

    summary_rows.append({
        "id": cve_id,
        "year": year,
        "base_score": base_score,
        "vendors": ", ".join(sorted(vendors)),
        "products": ", ".join(sorted(products)),
        "cpe_count": len(cpes),
        "vulnerable_cpes": "; ".join(vulnerable_cpes) if vulnerable_cpes else None,
        "snippet": en_desc[:120],
    })

summary_df = pd.DataFrame(summary_rows)

print(f"Summarized {len(summary_df)} CVEs")
summary_df

In [ ]:
cpe_rows = []

for fname in saved_files:
    with open(os.path.join(output_dir, fname), "r") as f:
        cve = json.load(f)

    cve_id = cve.get("id", "unknown")
    for c in extract_cpes(cve):
        vendor, product = parse_cpe_string(c["criteria"])
        cpe_rows.append({
            "id": cve_id,
            "vendor": vendor,
            "product": product,
            "criteria": c["criteria"],
            "vulnerable": c["vulnerable"],
            "versionEndIncluding": c["versionEndIncluding"],
            "versionEndExcluding": c["versionEndExcluding"],
        })

cpe_df = pd.DataFrame(cpe_rows)

print(f"{len(cpe_df)} CPE rows across {cpe_df['id'].nunique()} CVEs")
cpe_df

In [ ]:
pd.set_option("display.max_colwidth", None)

cpe_df["criteria"]